In [0]:
%run "/Workspace/Local To databrick manish migration/src/utility/logging_config"

In [0]:
logger.info(f" *********** List of folders in bucket (s3://de-manish-project/)***************")

for path in dbutils.fs.ls("s3://de-manish-project/"):
    print(path.path)

In [0]:
csv=[]
error_files=[]
source=dbutils.fs.ls("s3://de-manish-project/sales_data/")

if source:
    for path in source:
        if path.path.endswith(".csv"):
            csv.append(path.path)
        else:
            error_files.append(path.path)
    logger.info(f"here list of CSV {csv}")
    logger.info(f"here list of error {other_files}")
else:
    logger.error("There is no data to Process")
    raise Exception("There is no data to process.")



In [0]:
# %sql
# create table main_optimization_table.main_tables.product_staging_table(
#   id INT,
#   File_Name STRING,
#   file_location STRING,
#   Created_date timestamp,
#   Updated_date timestamp,
#   Status STRING

# )

In [0]:

files_names=[]
if csv:
    for f in csv:
        name=f.split("/")[-1]
        files_names.append(f"'{name}'")
    try:
        data = spark.sql(f"""select distinct File_Name,status
                    from main_optimization_table.main_tables.product_staging_table
                    where File_Name in ({','.join(files_names)})  AND status = 'File In Progress'
                    """)
    except Exception as e:
        logger.error(f"Error in staging {e}")
    # data = spark.sql(f"""select distinct File_Name
    #                  from main_optimization_table.main_tables.product_staging_table
    #                  where File_Name in ({','.join(files_names)})  AND status = 'File In Progress'
    #                  """)
    data.show()
    if data.count() > 0:
        logger.info("Please check last run")
    else:
        logger.info("No file in progress")
else:
    logger.info("No CSV file to process")




In [0]:
# files_names=[]
# for f in csv:
#     name=f.split("/")[-1]
#     files_names.append(f"'{name}'")

# files_names


In [0]:
# %sql
# INSERT INTO main_optimization_table.main_tables.product_staging_table
# VALUES
#   (2, 'sales_data.csv', 's3://de-manish-project/sales_data/sales_data.csv', from_utc_timestamp(current_timestamp(), 'Asia/Kolkata'), from_utc_timestamp(current_timestamp(), 'Asia/Kolkata'), 'File In Progress');

In [0]:
# %sql
# drop table  main_optimization_table.main_tables.product_staging_table 

In [0]:
%sql
Select * from main_optimization_table.main_tables.product_staging_table

In [0]:
csv

In [0]:
%run "/Workspace/Local To databrick manish migration/Resources/Dev/Con_note"

In [0]:
correct_files=[]
for files in csv:
    data_schema = spark.read.format("csv").option("header", "true").option("inferSchema", "true").load(files).columns
    logger.info(f"Schema of {files} is {data_schema}")
    logger.info(f"Mandatory columns schema is {mandatory_columns}")
    missing_columns = set(mandatory_columns) - set(data_schema)
    logger.info(f"Missing columns are {missing_columns}")

    if missing_columns:
        error_files.append(files)
    else:
        correct_files.append(files)
logger.info(f" *********** List of correct files after checking schema *************** {correct_files}")
logger.info(f" *********** List of error files checking schema*************** {error_files}")


    

In [0]:

correct_files

In [0]:
error_files

In [0]:
logger.info("******* Moving Error data to error directory if any ************")

error_distination="s3://de-manish-project/sales_data_error/"

for file in error_files:
    dbutils.fs.mv(file, error_distination)
    logger.info(f"Moving{file}------->{error_distination}  ************")


logger.info("******* Moved successfully ************")


In [0]:
cust = spark.read.format("delta") \
    .load("s3://de-manish-project/customer_data_mart")

In [0]:
cust.display()

In [0]:
sal_mart = spark.read.format("delta") \
    .load("s3://de-manish-project/customer_data_mart/")

sal_mart.display()

In [0]:
data = spark.read.format("csv") \
    .load("s3://de-manish-project/customer_data_mart/")

data.display()